# Planning Within the Battery

## Introduction

A spacecraft draws power all the time and its panels generate whenever they are
lit. A space telescope with a fixed, side-mounted array rolls it towards the Sun
while it observes, so in sunlight an observation puts back more than it draws. In
eclipse nothing comes back, and an observation there draws from the battery alone.
How much of each eclipse can be observed is then a question of what the battery
holds and how far it may be run down. A plan that ignores the battery observes
through every eclipse until the battery is past its discharge limit, and `DITL`,
which executes a plan exactly, does not charge on its own.

COASTSim's schedulers plan within the battery. Every plan keeps the battery above
`battery.planning_level`, a fraction of full that defaults to the depth-of-discharge
floor, `1 - max_depth_of_discharge`:

* The scheduler's ledger counts the deficit, the energy below full, step by step as
  the simulation does. The panels generate whenever they are lit, whatever the
  spacecraft is doing, so every span is counted at the attitude held through it:
  a slew along its path, an observation at its attitude, a contact along its
  tracking profile, and idle time at the attitude the spacecraft is left in. A
  full battery wastes what its panels make.
* The scheduler puts a `CHARGE` entry, pointing the panels at the Sun, into every
  idle sunlit stretch a charge fits, as it builds the plan, and counts what those
  charges put back. A snapshot goes only where the deficit then stays within the
  limit to the end of the day. Where it does not, the scheduler tries it later in
  the same gap, after the charging time that leaves, before trying later gaps.
  Snapshots are never shortened to fit, and requests are placed in merit order, so
  when charge runs short it is the lowest-merit science that waits.
* `DITL` executes the charges as it does contacts, slewing to the charging attitude
  and holding it, and checks them when it validates the plan.

This notebook plans a day for a spacecraft with a side-mounted array and a battery
that cannot carry a whole eclipse of observing. It plans the day with and without
the level, executes each plan, and shows what happens to the battery and to the
science.

## Step 1: Import Libraries

In [ ]:
%matplotlib inline
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
from rust_ephem import EarthLimbConstraint, SunConstraint, TLEEphemeris

from conops import DITL, RollingHorizonDITL
from conops.common import ACSMode, ObsType
from conops.config import (
    AttitudeControlSystem,
    Battery,
    Constraint,
    GroundStationRegistry,
    MissionConfig,
    Payload,
    PowerDraw,
    RadiatorConfiguration,
    SolarPanel,
    SolarPanelSet,
    SpacecraftBus,
    StarTrackerConfiguration,
    Telescope,
    TelescopeConfig,
    TelescopeType,
)
from conops.schedulers import CpSatScheduler, LocalSearchScheduler, PriorityScheduler
from conops.targets import Pointing

## Step 2: Spacecraft Configuration and Ephemeris

A spacecraft with one imaging telescope, the default ground stations and a 400 Wh
battery with a 30% depth of discharge, so a plan may use 120 Wh of it. The bus draws
120 W observing, 150 W slewing, 80 W idle and 60 W charging; the imager draws 150 W
observing and 50 W otherwise. One 600 W panel is mounted on the side of the
spacecraft. The scheduler gives every observation the power-optimal roll, which
turns the panel towards the Sun, so in sunlight an observation generates around
500 W against the 270 W it draws.

In eclipse nothing is generated. A 35-minute eclipse draws 158 Wh with the telescope
observing throughout and 76 Wh with it idle, so the battery can carry part of each
eclipse's observing but not all of it. Pointing and the rest are kept simple, as in
the scheduler tests: no star trackers or radiators, and Sun and Earth-limb
constraints only.

In [ ]:
IMAGER = "Imager"
WATTHOUR = 400.0
PANEL = 600.0


def make_config(normal: tuple[float, float, float]) -> MissionConfig:
    config = MissionConfig(
        ground_stations=GroundStationRegistry.default(),
        constraint=Constraint(
            sun_constraint=SunConstraint(min_angle=45),
            earth_constraint=EarthLimbConstraint(min_angle=10),
        ),
        solar_panel=SolarPanelSet(
            panels=[SolarPanel(name="Panel", normal=normal, max_power=PANEL)]
        ),
        battery=Battery(watthour=WATTHOUR, max_depth_of_discharge=0.3),
        spacecraft_bus=SpacecraftBus(
            attitude_control=AttitudeControlSystem(
                max_slew_rate=1.0, slew_acceleration=0.5, settle_time=30.0
            ),
            star_trackers=StarTrackerConfiguration(
                star_trackers=[], min_functional_trackers=0, modes_require_lock=[]
            ),
            radiators=RadiatorConfiguration(radiators=[]),
            power_draw=PowerDraw(
                nominal_power=80.0,
                power_mode={
                    int(ACSMode.SCIENCE): 120.0,
                    int(ACSMode.SLEWING): 150.0,
                    int(ACSMode.CHARGING): 60.0,
                },
            ),
        ),
        payload=Payload(
            instruments=[
                Telescope(
                    name=IMAGER,
                    optics=TelescopeConfig(
                        aperture_m=0.5,
                        focal_length_m=3.5,
                        telescope_type=TelescopeType.RITCHEY_CHRETIEN,
                        tube_length_m=1.0,
                    ),
                    power_draw=PowerDraw(
                        nominal_power=50.0, power_mode={int(ACSMode.SCIENCE): 150.0}
                    ),
                )
            ]
        ),
    )
    config.random_seed = 1234
    return config


begin = datetime(2025, 11, 1, 5, tzinfo=timezone.utc)
end = begin + timedelta(days=1)
ephemeris = TLEEphemeris(begin=begin, end=end, step_size=60, tle="example.tle")

config = make_config(normal=(0.0, 1.0, 0.0))
config.constraint.ephem = ephemeris
print(
    f"Planning level: {config.battery.planning_level:.0%} of {WATTHOUR:.0f} Wh, "
    f"so a plan may draw {config.battery.planning_deficit_wh:.0f} Wh"
)

## Step 3: Create the Targets

Sixty targets at random positions clear of the Sun, each wanting an hour of exposure
in 20-minute snapshots, with merits spread from 20 to 90. Together they ask for far
more than a day holds. How many of them are observed in the eclipses depends on the
battery.

In [ ]:
SNAPSHOT = 1200  # s


def make_targets(config: MissionConfig, seed: int) -> list[Pointing]:
    rng = np.random.default_rng(seed)
    targets = []
    for i in range(60):
        target = Pointing(
            config=config,
            ra=rng.uniform(0, 360),
            dec=rng.uniform(-60, 60),
            obsid=10000 + i,
            name=f"Target {i}",
            merit=rng.uniform(20, 90),
            ss_min=SNAPSHOT,
            ss_max=SNAPSHOT,
        )
        target.exptime = 3 * SNAPSHOT
        targets.append(target)
    return targets


targets = make_targets(config, 7)

## Step 4: Plan the Day With and Without the Level

`PriorityScheduler` places requests in merit order, each at the earliest time it
fits. With the level, "fits" includes the battery: a snapshot goes only where the
deficit stays within 120 Wh from there to the end of the day, counting the charges
the scheduler puts into the idle sunlit time.

To plan without regard to the battery, set `planning_level=None`.

In [ ]:
def plan_day(config, targets, level, label):
    config.battery.planning_level = level
    # A simulation drains the configured battery, so start each plan full.
    config.battery.charge_level = config.battery.watthour
    scheduler = PriorityScheduler(config, targets, begin, end)
    plan = scheduler.schedule()
    science = [e for e in plan if e.obstype == ObsType.AT]
    charges = [e for e in plan if e.obstype == ObsType.CHARGE]
    charging = sum(e.end - e.begin - e.slewtime for e in charges) / 3600
    print(
        f"{label:<16} {len(science):3d} snapshots, "
        f"{sum(e.exposure for e in science) / 3600:5.2f} h of science, "
        f"{len(charges):2d} charges, {charging:4.2f} h charging"
    )
    return scheduler, plan


limited_scheduler, limited_plan = plan_day(config, targets, 0.7, "With level")
unlimited_scheduler, unlimited_plan = plan_day(config, targets, None, "Without level")
config.battery.planning_level = 0.7

## Step 5: What the Scheduler Expects the Battery to Do

The scheduler's ledger of the battery, by orbit: each segment starts at a sunrise.
For each, the science placed in it, what the charges planned in it put back, and the
lowest battery level reached in it, at the end of the eclipse. The plan keeps every
low point above 70%; segments that touch it are where the battery, not the sky,
decided how much of the eclipse was observed. With a side-mounted array the charges
matter less than the observing itself, which charges the battery in sunlight: the
log says which requests had to wait for charge.

In [ ]:
ledger = limited_scheduler._battery
# The scheduler's accounting of the finished plan: every activity, charge and
# idle hold at the attitude flown.
energy = limited_scheduler._timeline_energy(
    limited_scheduler._charged_timeline(limited_scheduler.timeline)
)
levels = ledger.charge_levels(energy)
charges = [e for e in limited_plan if e.obstype == ObsType.CHARGE]
print(
    f"{'segment':>7} {'sunrise':>8} {'science':>8} {'charge':>9} {'lowest level':>13}"
)
for k in range(ledger.segments):
    start = (ledger.bounds[k] - begin.timestamp()) / 3600
    science = (
        sum(
            e.exposure
            for e in limited_plan
            if e.obstype == ObsType.AT and ledger.segment(float(e.begin)) == k
        )
        / 60
    )
    put_back = sum(
        ledger.net_charge(
            (e.ra, e.dec, e.roll), float(e.begin) + e.slewtime, float(e.end)
        )
        for e in charges
        if ledger.segment(float(e.begin)) == k
    )
    print(
        f"{k:7d} {start:7.2f}h {science:5.0f} min {put_back:6.0f} Wh {levels[k]:12.1%}"
        + ("  <- at the level" if levels[k] < 0.72 else "")
    )
waited = {
    e.obsid
    for e in limited_scheduler.log.events
    if "Not enough charge" in e.description
}
print(f"{len(waited)} requests waited for charge at some point")

## Step 6: Execute Both Plans

Each plan is executed by a `DITL`, which drains the battery at the power each mode
draws and charges it with what the panel generates at the attitude flown.
`validate_plan_matches_execution()` is empty when the simulation did exactly what the
plan said, charges included.

In [ ]:
def execute(config, plan, charge_fraction=1.0) -> DITL:
    config.battery.charge_level = charge_fraction * config.battery.watthour
    ditl = DITL(config=config, plan=plan, ephem=ephemeris, begin=begin, end=end)
    ditl.step_size = 60
    ditl.calc()
    return ditl


limited = execute(config, limited_plan)
unlimited = execute(config, unlimited_plan)


def report(label, run, level=0.7):
    battery = np.asarray(run.batterylevel)
    below = (battery < level - 1e-9).sum() * run.step_size / 3600
    print(
        f"{label:<16} lowest {battery.min():5.1%}, "
        f"{below:4.1f} h below the level, "
        f"{len(run.validate_plan_matches_execution()):3d} mismatches"
    )


report("With level", limited)
report("Without level", unlimited)

## Step 7: The Battery Through the Day

Battery level for each plan, with the planned charges and the eclipses shaded. Both
plans fill the battery in sunlight, observing. The plan with the level dips to 70%
through the eclipses and no further: it observes as much of each eclipse as the
battery can carry and leaves the rest idle. The plan without it observes whenever
the sky allows and runs the battery past its discharge limit in every eclipse.

In [ ]:
def plot_battery(ax, run, label, color):
    hours = (np.asarray(run.utime) - begin.timestamp()) / 3600
    ax.plot(hours, np.asarray(run.batterylevel) * 100, color=color, label=label)


def shade(ax, plan, run):
    first = True
    for e in plan:
        if e.obstype == ObsType.CHARGE:
            b = (float(e.begin) + e.slewtime - begin.timestamp()) / 3600
            f = (float(e.end) - begin.timestamp()) / 3600
            ax.axvspan(
                b, f, color="tab:orange", alpha=0.3, label="charging" if first else None
            )
            first = False
    hours = (np.asarray(run.utime) - begin.timestamp()) / 3600
    eclipse = np.array(
        [
            config.constraint.in_eclipse(ra=0.0, dec=0.0, time=float(t))
            for t in run.utime
        ]
    )
    ax.fill_between(
        hours, 0, 100, where=eclipse, color="black", alpha=0.06, label="eclipse"
    )


fig, ax = plt.subplots(figsize=(10, 4))
shade(ax, limited_plan, limited)
plot_battery(ax, limited, "With level", "tab:blue")
plot_battery(ax, unlimited, "Without level", "tab:red")
ax.axhline(
    70,
    color="tab:blue",
    ls="--",
    lw=1,
    label="planning level (the discharge floor, 70%)",
)
ax.set_xlim(0, 24)
ax.set_ylim(0, 102)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Battery level (%)")
ax.legend(ncol=3, fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

## Step 8: Who Got Observed

Exposure planned for each target, highest merit first, in each plan. Both plans take
the highest-merit targets first. With the level, fewer targets get their full hour,
and it is the lowest-merit ones that go without: the eclipse time the battery cannot
carry is taken from the end of the merit list.

In [ ]:
def planned_seconds(plan):
    seconds = {}
    for e in plan:
        if e.obstype == ObsType.AT:
            seconds[e.obsid] = seconds.get(e.obsid, 0.0) + e.exposure
    return seconds


by_merit = sorted(targets, key=lambda t: t.merit, reverse=True)
x = np.arange(len(by_merit))
fig, ax = plt.subplots(figsize=(10, 3.5))
for offset, (label, plan) in zip(
    (-0.2, 0.2), (("Without level", unlimited_plan), ("With level", limited_plan))
):
    seconds = planned_seconds(plan)
    ax.bar(
        x + offset, [seconds.get(t.obsid, 0) / 60 for t in by_merit], 0.4, label=label
    )
ax.set_xlabel("Targets, highest merit first")
ax.set_ylabel("Planned exposure (min)")
ax.legend()
fig.tight_layout()
plt.show()

## Step 9: Local Search Within the Level

`LocalSearchScheduler` starts from the priority-first plan and tries changes to its
order of snapshots, keeping those that raise merit-weighted science: each snapshot's
merit times its science seconds. Every plan it considers is decoded through the same
battery check, charges and all, so what it searches for is the best use of the
energy the day gives.

The priority-first plan placed the highest-merit requests first and left the
eclipses to whatever lower-merit snapshots still fit. Local search trades snapshots
between orbits and in and out of the eclipses on merit, and finds a little more
science and merit within the same level.

In [ ]:
def plan_with(scheduler_class, label, **options):
    config.battery.charge_level = config.battery.watthour
    scheduler = scheduler_class(config, targets, begin, end, **options)
    plan = scheduler.schedule()
    science = [e for e in plan if e.obstype == ObsType.AT]
    charges = [e for e in plan if e.obstype == ObsType.CHARGE]
    print(
        f"{label:<14} {len(science):3d} snapshots, "
        f"{sum(e.exposure for e in science) / 3600:5.2f} h of science, "
        f"{len(charges):2d} charges"
    )
    return scheduler, plan


local_scheduler, local_plan = plan_with(
    LocalSearchScheduler, "Local search", max_iterations=600, time_limit=600, seed=1
)
print(
    f"Merit-weighted science: {local_scheduler.initial_score[0] / 3600:.0f} -> "
    f"{local_scheduler.score[0] / 3600:.0f} merit-hours "
    f"({local_scheduler.iterations} changes tried, {local_scheduler.accepted} kept)"
)
local = execute(config, local_plan)
report("Local search", local)

## Step 10: CP-SAT Without the Battery in Its Model

`CpSatScheduler` solves each chunk of the day as a constraint model and decodes the
solver's answer through the same step as local search, so its plans are within the
level too. But the battery is not yet in its model: the solver, seeing only the sky,
fills the eclipses with science, the decoder refuses what the battery cannot carry,
and every chunk's solution loses to the priority-first plan it started from, which
is what it then keeps. Putting the battery into the model, as the recorder is, is
what it would take for the solver to trade eclipse science on energy directly.

In [ ]:
cpsat_scheduler, cpsat_plan = plan_with(
    CpSatScheduler, "CP-SAT", solver_time_limit=10.0, max_iterations=0, seed=1
)
print(
    f"Merit-weighted science: {cpsat_scheduler.start_score[0] / 3600:.0f} -> "
    f"{cpsat_scheduler.solver_score[0] / 3600:.0f} merit-hours; "
    f"{sum(cpsat_scheduler.solver_chunks_used)} of {len(cpsat_scheduler.solver_chunks_used)} "
    f"chunks improved, statuses {sorted(set(cpsat_scheduler.solver_statuses))}"
)
cpsat = execute(config, cpsat_plan)
report("CP-SAT", cpsat)

fig, ax = plt.subplots(figsize=(10, 3))
shade(ax, local_plan, local)
plot_battery(ax, limited, "Priority", "tab:blue")
plot_battery(ax, local, "Local search", "tab:green")
ax.axhline(70, color="tab:blue", ls="--", lw=1)
ax.set_xlim(0, 24)
ax.set_ylim(50, 102)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Battery level (%)")
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

## Step 11: Keeping Margin

The level is a planning choice. A higher one keeps more charge back, for an eclipse
that runs long or a charge that is cut short, at the cost of science: at 72% the
plan may draw 112 Wh instead of 120, so it observes less of each eclipse. It has a
floor of its own: an idle spacecraft draws 76 Wh through each eclipse whatever the
plan does, plus the slews around it, so at 75% the 100 Wh allowed is less than the
spacecraft draws just getting through the night by the scheduler's reckoning. No
plan can hold that level, so the scheduler places nothing and says so in its log.

In [ ]:
margin_scheduler, margin_plan = plan_day(config, targets, 0.72, "72% level")
margin = execute(config, margin_plan)
report("72% level", margin, level=0.72)

tight_scheduler, tight_plan = plan_day(config, targets, 0.75, "75% level")
tight = execute(config, tight_plan)
report("75% level", tight, level=0.75)
for event in tight_scheduler.log.events:
    if "goes below" in event.description:
        print(event.description)
config.battery.planning_level = 0.7

## Step 12: Rolling-Horizon Replanning

`RollingHorizonDITL` plans a few hours at a time and replans as the day goes, each
plan starting from the battery as it will be when the plan takes over: its charge
now, plus what the activities already committed draw and put back until then. Here
the run starts with the battery at 80%, so the first plan has less to spend, and
every replan keeps the day above the level.

In [ ]:
config.battery.charge_level = 0.8 * config.battery.watthour
rolling = RollingHorizonDITL(
    config,
    make_targets(config, 7),
    ephem=ephemeris,
    begin=begin,
    end=end,
    horizon=timedelta(hours=6),
    replan_interval=timedelta(hours=3),
    commit_lead_time=timedelta(minutes=15),
)
rolling.step_size = 60
rolling.calc()
print(f"{len(rolling.replans)} plans built")
report("Rolling", rolling)
charges = [e for e in rolling.plan if e.obstype == ObsType.CHARGE]
print(f"{len(charges)} charges across the replans")

fig, ax = plt.subplots(figsize=(10, 3))
shade(ax, rolling.plan, rolling)
plot_battery(ax, rolling, "Rolling, from 80%", "tab:purple")
for record in rolling.replans:
    ax.axvline((record.utime - begin.timestamp()) / 3600, color="gray", lw=0.5, ls=":")
ax.axhline(70, color="tab:blue", ls="--", lw=1)
ax.set_xlim(0, 24)
ax.set_ylim(0, 102)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Battery level (%)")
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

## Summary

* `battery.planning_level` is the level every plan keeps the battery above; it
  defaults to the depth-of-discharge floor.
* The scheduler's ledger counts energy as the simulation does, step by step at the
  attitude held, since the panels generate whenever they are lit; the plan charges
  in every idle sunlit stretch a charge fits.
* A snapshot goes only where the battery can carry it to the end of the day, and
  waits for charge otherwise; `DITL` executes the charges with the rest of the plan.
* Local search trades snapshots on merit within the same level; CP-SAT's plans are
  within it too, but its model does not yet know the battery, so it cannot gain.
* With a side-mounted array the battery binds in the eclipses: the plan observes as
  much of each as the battery can carry, and it is the lowest-merit science that
  gives way.